# Three Beginner Data Science Projects
**Author:** Joseph Cosmos Aniebiet (22/SC/CO/1301)

1. **Titanic survival prediction** (binary classification)
2. **House price prediction** (regression, California Housing)
3. **Iris flower classification** (multi-class classification)

Each project follows the same workflow: define the question, load data, clean it, explore it, engineer
features, train a model, evaluate it, and communicate the result.

In [1]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
FIG = Path("../figures"); FIG.mkdir(exist_ok=True)
NAVY, BLUE, TEAL, AMBER, RED, GREY = "#0B2A6F", "#1F5FD6", "#14A38B", "#E8A33D", "#D64545", "#8A94A6"
PALETTE = [NAVY, BLUE, TEAL, AMBER, RED, "#7B61FF", GREY]
sns.set_theme(style="whitegrid", palette=PALETTE, font="DejaVu Sans")
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "axes.titleweight": "bold",
                     "axes.titlesize": 13, "axes.labelsize": 11, "axes.spines.top": False,
                     "axes.spines.right": False})
def save(name):
    plt.tight_layout()
    plt.savefig(FIG / f"{name}.png", bbox_inches="tight", facecolor="white")
    plt.show()

In [2]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             mean_absolute_error, mean_squared_error, r2_score, ConfusionMatrixDisplay)
from sklearn.datasets import load_iris

## Project 1: Titanic survival prediction
**Question:** given a passenger's details, did they survive?

In [3]:
titanic = pd.read_csv("../data/titanic.csv")
print(titanic.shape)
titanic.isna().sum()[lambda s: s > 0]

(891, 15)


age            177
embarked         2
deck           688
embark_town      2
dtype: int64

In [4]:
df = titanic[["survived", "pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]].copy()
df["age"] = df.age.fillna(df.groupby(["pclass", "sex"]).age.transform("median"))   # impute by group
df["embarked"] = df.embarked.fillna(df.embarked.mode()[0])
df["family_size"] = df.sibsp + df.parch + 1                                           # feature engineering
df["is_child"] = (df.age < 13).astype(int)
df["sex"] = df.sex.map({"male": 0, "female": 1})
df = pd.get_dummies(df, columns=["embarked"], drop_first=True, dtype=int)
fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
sns.barplot(data=titanic, x="sex", y="survived", hue="sex", legend=False, ax=ax[0], palette=[BLUE, RED], errorbar=None)
ax[0].set(title="Survival rate by sex", ylabel="Survival rate", xlabel="")
sns.barplot(data=titanic, x="pclass", y="survived", hue="pclass", legend=False, ax=ax[1], palette=[NAVY, BLUE, TEAL], errorbar=None)
ax[1].set(title="Survival rate by class", ylabel="", xlabel="Passenger class")
sns.histplot(data=titanic, x="age", hue="alive", bins=30, ax=ax[2], palette=[RED, TEAL], multiple="stack")
ax[2].set(title="Age distribution by outcome", xlabel="Age")
save("bp_titanic_eda")

In [5]:
X, y = df.drop(columns="survived"), df.survived
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
models = {"Logistic Regression": LogisticRegression(max_iter=1000),
          "Decision Tree": DecisionTreeClassifier(max_depth=4, random_state=42),
          "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42)}
rows = []
for name, m in models.items():
    m.fit(X_tr, y_tr)
    rows.append({"model": name, "test_accuracy": accuracy_score(y_te, m.predict(X_te)),
                 "cv_accuracy": cross_val_score(m, X, y, cv=5).mean()})
titanic_results = pd.DataFrame(rows).round(3)
best = models["Random Forest"]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ConfusionMatrixDisplay.from_estimator(best, X_te, y_te, display_labels=["Died", "Survived"], cmap="Blues", ax=ax[0], colorbar=False)
ax[0].set_title("Random Forest confusion matrix"); ax[0].grid(False)
imp = pd.Series(best.feature_importances_, index=X.columns).sort_values()
ax[1].barh(imp.index, imp.values, color=NAVY); ax[1].set_title("Feature importance")
save("bp_titanic_model")
titanic_results

,model,test_accuracy,cv_accuracy
0,Logistic Regression,0.816,0.807
1,Decision Tree,0.799,0.807
2,Random Forest,0.804,0.822


## Project 2: House price prediction
**Question:** what is the median house value of a district, given its characteristics?

In [6]:
housing = pd.read_csv("../data/housing.csv")
print(housing.shape); print(housing.isna().sum()[lambda s: s > 0])
h = housing.copy()
h["total_bedrooms"] = h.total_bedrooms.fillna(h.total_bedrooms.median())
h["rooms_per_household"] = h.total_rooms / h.households
h["bedrooms_per_room"] = h.total_bedrooms / h.total_rooms
h["population_per_household"] = h.population / h.households
h = pd.get_dummies(h, columns=["ocean_proximity"], dtype=int)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.3))
sc = ax[0].scatter(housing.longitude, housing.latitude, c=housing.median_house_value, s=housing.population / 200,
                   cmap="viridis", alpha=0.5)
plt.colorbar(sc, ax=ax[0], label="Median house value ($)")
ax[0].set(title="House value by location", xlabel="Longitude", ylabel="Latitude")
ax[1].scatter(housing.median_income, housing.median_house_value, s=4, alpha=0.2, color=BLUE)
ax[1].set(title="Income vs. house value", xlabel="Median income (tens of thousands $)", ylabel="Median house value ($)")
save("bp_housing_eda")

(20640, 10)
total_bedrooms    207
dtype: int64


In [7]:
X, y = h.drop(columns="median_house_value"), h.median_house_value
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
reg = {"Linear Regression": LinearRegression(),
       "Random Forest": RandomForestRegressor(n_estimators=150, n_jobs=-1, random_state=42)}
rows, preds = [], {}
for name, m in reg.items():
    m.fit(X_tr, y_tr); p = m.predict(X_te); preds[name] = p
    rows.append({"model": name, "MAE": mean_absolute_error(y_te, p),
                 "RMSE": mean_squared_error(y_te, p) ** 0.5, "R2": r2_score(y_te, p)})
housing_results = pd.DataFrame(rows).round(3)
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(y_te, preds["Random Forest"], s=5, alpha=0.3, color=TEAL)
lim = [0, 520000]; ax.plot(lim, lim, color=RED, lw=2, label="Perfect prediction")
ax.set(xlim=lim, ylim=lim, title="Random Forest: predicted vs. actual", xlabel="Actual value ($)", ylabel="Predicted value ($)")
ax.legend(frameon=False)
save("bp_housing_model")
housing_results

,model,MAE,RMSE,R2
0,Linear Regression,50888.66,72668.538,0.597
1,Random Forest,32230.29,50131.430,0.808


## Project 3: Iris flower classification
**Question:** which of three iris species is a flower, from four measurements?

In [8]:
iris = load_iris(as_frame=True)
idf = iris.frame.copy(); idf["species"] = idf.target.map(dict(enumerate(iris.target_names)))
g = sns.pairplot(idf.drop(columns="target"), hue="species", palette=[NAVY, TEAL, AMBER], height=2.0,
                 plot_kws={"s": 18, "alpha": 0.8})
g.figure.suptitle("Iris measurements by species", y=1.02, fontweight="bold")
g.savefig(FIG / "bp_iris_eda.png", dpi=170, bbox_inches="tight", facecolor="white")

In [9]:
X, y = iris.data, iris.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
knn = {k: accuracy_score(y_te, KNeighborsClassifier(n_neighbors=k).fit(X_tr, y_tr).predict(X_te)) for k in range(1, 21)}
best_k = max(knn, key=knn.get)
model = KNeighborsClassifier(n_neighbors=best_k).fit(X_tr, y_tr)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(list(knn), list(knn.values()), marker="o", color=NAVY); ax[0].axvline(best_k, color=RED, ls="--")
ax[0].set(title="Choosing k for K-Nearest Neighbours", xlabel="k (neighbours)", ylabel="Test accuracy")
ConfusionMatrixDisplay.from_estimator(model, X_te, y_te, display_labels=iris.target_names, cmap="Greens", ax=ax[1], colorbar=False)
ax[1].set_title(f"KNN (k = {best_k}) confusion matrix"); ax[1].grid(False)
save("bp_iris_model")
print("best k:", best_k, "accuracy:", round(knn[best_k], 3))
print(classification_report(y_te, model.predict(X_te), target_names=iris.target_names))

best k: 5 accuracy: 0.978
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        15
  versicolor       0.94      1.00      0.97        15
   virginica       1.00      0.93      0.97        15

    accuracy                           0.98        45
   macro avg       0.98      0.98      0.98        45
weighted avg       0.98      0.98      0.98        45



In [10]:
import json
summary = {"titanic": titanic_results.to_dict("records"), "housing": housing_results.to_dict("records"),
           "iris": {"best_k": int(best_k), "accuracy": round(float(knn[best_k]), 3)}}
json.dump(summary, open(FIG / "beginner_results.json", "w"), indent=2)
summary

{'titanic': [{'model': 'Logistic Regression',
   'test_accuracy': 0.816,
   'cv_accuracy': 0.807},
  {'model': 'Decision Tree', 'test_accuracy': 0.799, 'cv_accuracy': 0.807},
  {'model': 'Random Forest', 'test_accuracy': 0.804, 'cv_accuracy': 0.822}],
 'housing': [{'model': 'Linear Regression',
   'MAE': 50888.66,
   'RMSE': 72668.538,
   'R2': 0.597},
  {'model': 'Random Forest', 'MAE': 32230.29, 'RMSE': 50131.43, 'R2': 0.808}],
 'iris': {'best_k': 5, 'accuracy': 0.978}}